In [ ]:
!pip install transformers sentence-transformers faiss-cpu newspaper3k bs4

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import torch
import faiss
import numpy as np

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
documents = [
    {
        "text": "Burnaby RCMP are investigating after a pedestrian was struck near Metrotown on Monday morning. Emergency crews responded around 7 a.m. and the victim was transported to hospital with non-life-threatening injuries.",
        "url": "https://www.cbc.ca/news/canada/british-columbia",
        "source": "CBC BC"
    },
    {
        "text": "TransLink says SkyTrain service on the Expo Line was delayed Tuesday afternoon due to a mechanical issue near Commercial–Broadway Station, causing significant congestion during the rush hour commute.",
        "url": "https://globalnews.ca/bc",
        "source": "Global News BC"
    },
    {
        "text": "Vancouver police say they responded to a break-and-enter in Kitsilano overnight, urging residents to secure windows and report suspicious behaviour as property crime rises in the area.",
        "url": "https://bc.ctvnews.ca",
        "source": "CTV Vancouver"
    },
    {
        "text": "The B.C. government announced a new housing investment plan aimed at expanding affordable rentals across Metro Vancouver, with construction expected to begin in early 2025.",
        "url": "https://news.gov.bc.ca",
        "source": "BC Gov News"
    }
]

In [ ]:
corpus_embeddings = embedder.encode([d["text"] for d in documents], convert_to_tensor=False)
corpus_embeddings = np.array(corpus_embeddings).astype("float32")

In [ ]:
index = faiss.IndexFlatL2(corpus_embeddings.shape[1])
index.add(corpus_embeddings)

In [ ]:
query = "What happened in Burnaby today?"
query_emb = embedder.encode(query)

D, I = index.search(np.array([query_emb]).astype("float32"), k=2)

retrieved = [documents[i] for i in I[0]]

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

context = "\n\n".join([f"[{d['source']}] {d['text']}" for d in retrieved])

prompt = (
    "You are a summarization system.\n"
    "You MUST summarize ONLY the information in the articles below.\n"
    "If something is NOT stated in the text, do NOT invent it.\n"
    "Include the EXACT sources provided.\n"
    "Do NOT guess or fabricate.\n\n"
    "ARTICLES:\n"
    f"{context}\n\n"
    "SUMMARY:"
)

inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)

output = gpt2.generate(
    **inputs,
    max_length=200,
    do_sample=False,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

context_lines = []
for d in retrieved:
    context_lines.append(f"- [{d['source']}] {d['text']}")
context = "\n".join(context_lines)

prompt = f"""
Summarize the following local news in 2–3 sentences.
Use ONLY the information provided.
Do NOT add anything that is not stated.
Mention the sources in parentheses at the end.

NEWS:
{context}

Summary:
""".strip()

inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)

output_ids = gpt2.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=False,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.eos_token_id,
)

generated_only = output_ids[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(generated_only, skip_special_tokens=True))